# MP34 — Satellite HR–LR Pair Builder

[Brief](README.md) · [Validation](VALIDATION.md). Objective: construct reproducible synthetic ×2/×4 pairs with compatible dimensions, transforms and masks. Prerequisite MP33. The example source is aerial NAIP; synthetic pairing does not establish real satellite sensor correspondence.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
import torch
from affine import Affine
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'stage_08_remote_sensing/MP34_satellite_hr_lr_pair_builder'))
from mp34_tools import make_pair,georeferenced_pair


## 1. The degradation equation

y=D_s(k*x)+n. HR x has C×H×W values; y has C×(H/s)×(W/s). The normalized Gaussian k uses σ=0.8 HR pixels, radius ceil(3σ), replicate boundaries; D_s takes a nonoverlapping s×s area mean. Noise n is optional seeded additive Gaussian in input units; the recorded real experiment uses noise0. No clipping is applied to scientific pair arrays. C, band ordering and numeric scaling stay fixed.

This is an explicit synthetic forward model, not a measured satellite point-spread function. Different real sensors can differ in spectral response, acquisition time, viewing angle, atmosphere, cloud cover and registration. Synthetic pairs isolate a controlled learning problem but cannot validate those differences.

In [ ]:
hr=torch.arange(16,dtype=torch.float32).reshape(1,4,4)
lr=make_pair(hr,2,sigma=0)
torch.testing.assert_close(lr,torch.tensor([[[2.5,4.5],[10.5,12.5]]]))
assert make_pair(hr,4,sigma=0).item()==7.5
print('HR:',hr.tolist(),'area x2:',lr.tolist())


## 2. Shape and footprint alignment

Hand calculation: the top-left2×2 block [0,1;4,5] averages to2.5. A4×4 image becomes2×2 at×2 or1×1 at×4. Require divisibility rather than silently crop or resize. If the HR pixel-to-map transform is T, use T_LR=T×Scale(s,s), keeping the CRS and upper-left corner. The four exterior map corners must agree. Pixel **centers** differ intentionally: each coarse sample summarizes an HR footprint.

For HR10-m north-up pixels and corner(100,200), the first HR center is(105,195); first×2 LR center is(110,190). Both cover the same4×4 HR outer extent. Rotated/skewed grids require the full affine multiplication.

In [ ]:
transform=Affine(10,2,100,1,-10,200)
lr,meta=georeferenced_pair(hr,transform,'EPSG:32632',scale=2,sigma=0)
assert meta['hr_corners']==meta['lr_corners']
assert meta['lr_transform']==[20,4,100,2,-20,200]
print('Shared corners:',meta['hr_corners'])
print('Transforms:',meta['hr_transform'],meta['lr_transform'])


## 3. Validity, noise and provenance

This bounded builder requires fully valid HR pixels. It rejects nodata instead of allowing an arbitrary fill to spread through the Gaussian kernel. For partially valid imagery, a separately specified support-aware degradation/mask protocol is needed; this implementation does not invent one. A valid mask is not a cloud mask. Split scenes geographically before making pairs, and preserve source license/checksum, crop offsets, bands, units, operator and seed.

Fixed seed gives identical noise for a repeated input/settings combination. It does not make the noise a physical sensor model. Unchanged source arrays and deterministic output hashes support reproducibility, not physical correctness.

In [ ]:
valid=torch.ones(4,4,dtype=torch.bool);valid[0,0]=False
try:georeferenced_pair(hr,transform,'EPSG:32632',valid=valid)
except ValueError as error:print('Expected invalid-pixel rejection:',error)
else:raise AssertionError('Invalid HR must be rejected')
a=make_pair(hr,noise=.01,seed=17);b=make_pair(hr,noise=.01,seed=17)
torch.testing.assert_close(a,b,rtol=0,atol=0)


## 4. Executed real-raster pairs

The experiment verifies the pinned training raster's SHA-256, reads a64×64 RGBNIR window at column238,row238, and divides stored uint16 DN by10000 without per-image stretching. σ=.8, noise0, and area×2/×4 produce32×32 and16×16 pairs. HR pixel spacing is2.5m; LR spacing is5m and10m. The common map footprint is160×160m in EPSG:32611. This describes the raster grid, not native optical resolution.

The manifest below records exact transforms, shapes, array-file hashes, operator and environment. Pair arrays live in ignored data/teaching/mp34 and can be rebuilt with `python experiment.py` from the pinned training source. Notebook execution reads the recorded manifest; it does not overwrite pairs or re-evaluate any model.

In [ ]:
record=json.loads((root/'reports/mp34-pairing-manifest.json').read_text())
assert record['source']['split']=='train'
for pair in record['rows']:
    assert pair['hr_corners']==pair['lr_corners']
    assert pair['hr_shape']==[4,64,64]
    assert pair['lr_shape']==[4,64//pair['scale'],64//pair['scale']]
    print('Scale',pair['scale'],'shape',pair['lr_shape'],'transform',pair['lr_transform'])
    print('Normalized-DN ranges:',pair['hr_range'],pair['lr_range'],'LR file hash',pair['lr_sha256'])
print(record['license'])


## 5. Exercises and separate solutions

1. Derive the top-left area×2 value of [0,1;4,5]. 2. A64×64 HR crop has2.5-m spacing: give LR shapes/spacings at×2 and×4. 3. Why not replace nodata with zero before blurring? 4. Does identical geographic extent prove accurate cross-sensor pairing?

<details><summary>Solutions</summary>

1. (0+1+4+5)/4=2.5. 2.32×32 at5m and16×16 at10m, both160m per side for this north-up grid. 3. The fill contaminates adjacent pixels through convolution; masks need support-aware treatment or the patch must be rejected. 4. No: spectral response, time, viewing geometry, registration and calibration can differ even with matching extents.
</details>

## Interpretation and connection

Both measured pairs have the intended shapes and exact matching exterior corners; smoothing/downsampling changes pixel ranges as recorded above. This is evidence of declared synthetic pairing, not improved SR fidelity. ×4 discards more samples and is not interchangeable with the hosted×2 models. Next MP35 checks geographic train/validation/test separation. Frozen test data, checkpoints and inference normalization remain unchanged; no learner mastery or real-satellite generalization claim.